# Telugu GPT-2 Evaluation on Colab

This notebook is designed for Google Colab with a T4 GPU. It loads the Hugging Face model repo, runs prompt-based generation checks, and computes perplexity on a Telugu evaluation text file.

## What This Notebook Covers

1. Install dependencies
2. Load `pulipakav-1/dravidian-gpt2-telugu`
3. Run a small Telugu prompt suite
4. Upload a `.txt` evaluation file and compute perplexity
5. Save results as JSON

In [ ]:
!pip -q install -U transformers datasets sentencepiece accelerate

In [ ]:
import json
import math
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_ID = "pulipakav-1/dravidian-gpt2-telugu"
OUTPUT_DIR = Path("/content/dravidian_eval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_NEW_TOKENS = 80
TEMPERATURE = 0.8
TOP_P = 0.95

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print({
    "device": device,
    "dtype": str(dtype),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)
model.to(device)
model.eval()

print("Loaded model:", MODEL_ID)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
prompts = [
    "తెలుగు భాష గురించి ఒక చిన్న పేరా రాయండి.",
    "విజయవాడ నగరంపై ఐదు వాక్యాలు రాయండి.",
    "వర్షపు రోజు గురించి ఒక చిన్న కథ మొదలు పెట్టండి.",
    "భవిష్యత్తులో విద్య ఎలా మారుతుంది అనే విషయంపై తెలుగులో రాయండి.",
    "కృషి మరియు రైతుల ప్రాముఖ్యతపై ఒక చిన్న వ్యాఖ్యానం రాయండి.",
]

generation_results = []
for i, prompt in enumerate(prompts, start=1):
    output = generate_text(prompt)
    generation_results.append({"id": i, "prompt": prompt, "output": output})
    print(f"\n--- Prompt {i} ---")
    print("PROMPT:")
    print(prompt)
    print("\nOUTPUT:")
    print(output)


## Perplexity Evaluation

Upload a UTF-8 Telugu `.txt` file. Best practice:

- one sentence or paragraph per line
- text not seen during training if possible
- at least a few thousand words for a stable estimate

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded

In [ ]:
if not uploaded:
    raise ValueError("No file uploaded.")

eval_filename = next(iter(uploaded))
eval_path = Path(eval_filename)
text = eval_path.read_text(encoding="utf-8")

lines = [line.strip() for line in text.splitlines() if line.strip()]
joined_text = "\n".join(lines)

print({
    "file": eval_filename,
    "num_lines": len(lines),
    "num_characters": len(joined_text),
})

In [ ]:
def compute_perplexity(text, model, tokenizer, stride=512):
    encodings = tokenizer(text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)
    max_length = model.config.n_positions
    nlls = []
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = encodings.input_ids[:, begin:end].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end = end
        if end == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / seq_len)
    return {
        "perplexity": float(ppl.item()),
        "num_tokens": int(seq_len),
        "stride": stride,
        "context_window": int(max_length),
    }

In [ ]:
perplexity_result = compute_perplexity(joined_text, model, tokenizer)
perplexity_result

In [ ]:
results = {
    "model_id": MODEL_ID,
    "device": device,
    "seed": SEED,
    "generation_config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
    },
    "prompts": generation_results,
    "perplexity": perplexity_result,
    "eval_file": eval_filename,
}

out_path = OUTPUT_DIR / "telugu_gpt2_eval_results.json"
out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved results to: {out_path}")

In [ ]:
from google.colab import files
files.download(str(out_path))